In [4]:
import random

class WarehouseACO:
    def __init__(self, distance_matrix, num_ants=10, generations=30, alpha=1.0, beta=2.0, evaporation=0.4, Q=100.0):
        self.distances = distance_matrix
        self.num_nodes = len(distance_matrix)
        self.num_ants = num_ants
        self.generations = generations

        # Hyperparameters
        self.alpha = alpha          # Pheromone importance
        self.beta = beta            # Distance importance
        self.evaporation = evaporation  # Evaporation coefficient
        self.Q = Q                  # Pheromone deposit factor

        # Initialize pheromone matrix evenly on all links
        self.pheromones = [[1.0 for _ in range(self.num_nodes)] for _ in range(self.num_nodes)]

        self.global_best_tour = None
        self.global_best_distance = float('inf')

    def _select_next_node(self, current_node, unvisited):
        """Calculates probabilities and picks the next node using a roulette wheel selection."""
        probabilities = []
        total_weight = 0.0

        for next_node in unvisited:
            pheromone = self.pheromones[current_node][next_node] ** self.alpha
            visibility = (1.0 / max(self.distances[current_node][next_node], 1e-6)) ** self.beta
            weight = pheromone * visibility
            probabilities.append((next_node, weight))
            total_weight += weight

        pick = random.uniform(0, total_weight)
        current_weight = 0.0
        for node, weight in probabilities:
            current_weight += weight
            if current_weight >= pick:
                return node
        return list(unvisited)[-1]

    def _simulate_ants(self):
        """Simulates all ants constructing their picking tours."""
        for _ in range(self.num_ants):
            current_node = 0
            unvisited = set(range(1, self.num_nodes))
            tour = [current_node]
            tour_distance = 0

            while unvisited:
                next_node = self._select_next_node(current_node, unvisited)
                tour_distance += self.distances[current_node][next_node]
                tour.append(next_node)
                unvisited.remove(next_node)
                current_node = next_node

            tour_distance += self.distances[current_node][0]
            tour.append(0)

            if tour_distance < self.global_best_distance:
                self.global_best_distance = tour_distance
                self.global_best_tour = list(tour)

    def _update_pheromones(self):
        """Handles matrix evaporation and deposits rewards on the global best path."""
        for i in range(self.num_nodes):
            for j in range(self.num_nodes):
                self.pheromones[i][j] *= (1.0 - self.evaporation)

        if self.global_best_tour:
            deposit_amount = self.Q / self.global_best_distance
            for idx in range(len(self.global_best_tour) - 1):
                from_node = self.global_best_tour[idx]
                to_node = self.global_best_tour[idx + 1]
                self.pheromones[from_node][to_node] += deposit_amount
                self.pheromones[to_node][from_node] += deposit_amount

    def run(self):
        """Executes the standard optimization routine."""
        for _ in range(self.generations):
            self._simulate_ants()
            self._update_pheromones()
        return self.global_best_tour, self.global_best_distance


# --- INITIALIZATION & EXECUTION ---
if __name__ == "__main__":
    # Formatted explicitly to prevent any truncation errors
    row0 = [0, 15, 25, 10, 30]  # Depot
    row1 = [15, 0, 12, 22, 18]  # Item 1
    row2 = [25, 12, 0, 35, 14]  # Item 2
    row3 = [10, 22, 35, 0, 20]  # Item 3
    row4 = [30, 18, 14, 20, 0]  # Item 4

    warehouse_distances = [row0, row1, row2, row3, row4]

    # Run optimizer
    aco_optimizer = WarehouseACO(
        distance_matrix=warehouse_distances,
        num_ants=15,
        generations=40,
        evaporation=0.3
    )

    best_sequence, minimal_distance = aco_optimizer.run()

    print("=== STANDARDIZED ACO PICKING ROUTE ===")
    print(f"Optimal Sequence Layout: {' -> '.join(map(str, best_sequence))}")
    print(f"Calculated Trip Distance : {minimal_distance} meters")


=== STANDARDIZED ACO PICKING ROUTE ===
Optimal Sequence Layout: 0 -> 3 -> 4 -> 2 -> 1 -> 0
Calculated Trip Distance : 71 meters
